# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [42]:
origine = 'datagouv_organization_or_owner'
priorite = 'priorite'
coord = 'coordonneesXY'
id_station = 'id_station_itinerance'
id_pdc = 'id_pdc_itinerance'
last_modif = 'last_modified'
date_maj = 'date_maj'
nom_station = 'nom_station'
adresse = 'adresse_station'
amenageur = 'nom_amenageur'

unicite_stations = [id_station, origine, date_maj, last_modif]
filtre = [priorite,  date_maj, last_modif]
id_station_pdc = [id_station, id_pdc]
att_station = [id_station, date_maj, last_modif, amenageur, nom_station, coord]
att_pdc = [id_pdc, id_station, date_maj, last_modif, amenageur, nom_station, coord]

In [4]:
file_irve_brut = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260301.csv' # données brutes 28/02
file_irve = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260401.csv' # données dédoublonnées 28/02

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut[last_modif] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve[last_modif] = irve['datagouv_last_modified']

## Données brutes

In [5]:
print('nombre de lignes : {}, nombre de pdc : {} \n'.format(len(irve_brut), len(irve_brut.groupby([id_pdc]).count())))
print(irve_brut.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_brut = analyse_integrite(irve_brut, schema)

nombre de lignes : 379946, nombre de pdc : 147826 

datagouv_organization_or_owner
QualiCharge                       61545
Engie Mobilités Electriques       58305
GIREVE                            31714
TotalEnergies Marketing France    29490
Mobilize Power Solutions          21033
IZIVIA                            15767
Driveco                           15551
ubitricity                        15349
Alizé                             14334
STATIONS-E                        13260
Name: index, dtype: int64 

index - id_pdc_itinerance                          295800
contact_operateur - id_station_itinerance          78950
nom_enseigne - id_station_itinerance               61760
coordonneesXY - id_station_itinerance              117807
id_station_itinerance - id_pdc_itinerance          150304
nom_station - id_station_itinerance                57457
implantation_station - id_station_itinerance       79455
nbre_pdc - id_station_itinerance                   73233
condition_acces - id_station_i

## Dédoublonnage actuel

In [6]:
print('nombre de lignes : {} \n'.format(len(irve)))
print(irve.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_actuel = analyse_integrite(irve, schema)

nombre de lignes : 148286 

datagouv_organization_or_owner
QualiCharge                                  46503
GIREVE                                       31714
Alizé                                        13624
IZIVIA                                       12958
Eco-Movement                                 10806
Indigo Group                                  6907
Driveco                                       3935
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1827
Citeos Ingénierie IdF & Est (Cogelum IdF)     1485
Name: index, dtype: int64 

index - id_pdc_itinerance                            816
contact_operateur - id_station_itinerance           1777
nom_enseigne - id_station_itinerance                4064
coordonneesXY - id_station_itinerance               5402
id_station_itinerance - id_pdc_itinerance             28
nom_station - id_station_itinerance                 2436
implantation_station - id_station_itinerance        2408
nbre

## Dédoublonnage proposé
Les critères utilisés pour éliminer les doublons sont par ordre : priorite (datagouv_organization_or_owner), date (date_maj, datagouv_last_modified)

In [7]:
# dédoublonnage des stations (suivant critères d'unicité)
stations = irve_brut.drop_duplicates(unicite_stations).copy()

print("nombre de stations brutes {}, stations suivant critère d'unicité {} et stations uniques {} ".format(len(irve_brut), len(stations), len(irve_brut.drop_duplicates(id_station))))

nombre de stations brutes 379946, stations suivant critère d'unicité 126827 et stations uniques 59045 


### Dédoublonnage direct station
A l'issue de cette étape, on a une liste de stations uniques respectant les critères de filtrage

avec l'origine la plus prioritaire et la mise à jour la plus récente

In [8]:
# choix du critère de priorité (Qualicharge)
stations[priorite] = stations[origine] == 'QualiCharge'

# dédoublonnage des stations suivant son id_station_itinerance avec filtrage suivant les critères retenus
stat_direct = stations.sort_values(by=[id_station] + filtre).drop_duplicates(id_station, keep='last').copy()
if stations[priorite].sum() != stat_direct[priorite].sum():
    print('priorité non respectée')

In [9]:
print('nombre de stations initiales {}, stations dédoublonnées {} (supprimées {})'.format(len(stations), len(stat_direct), len(stations) - len(stat_direct)))
stat_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

nombre de stations initiales 126827, stations dédoublonnées 59045 (supprimées 67782)


datagouv_organization_or_owner
QualiCharge                                  13597
Eco-Movement                                 10485
GIREVE                                        9873
IZIVIA                                        6656
Alizé                                         4483
GREENEA                                       1910
Load Stations                                 1091
Driveco                                        892
Syndicat Départemental d'Energie du Tarn       700
Citeos Ingénierie IdF & Est (Cogelum IdF)      616
ZE-WATT                                        607
SOREGIES                                       451
Electric 55 Charging                           391
ZEborne                                        377
Mobilize Power Solutions                       373
Name: index, dtype: int64

### Dédoublonnage direct pdc
A l'issue de cette étape, chaque pdc est présent une seule fois sur la station respectant les critères de filtrage

In [10]:
# dédoublonnage des pdc présents sur plusieurs stations
pdc_stat = stat_direct[unicite_stations + [priorite]].merge(irve_brut, how='left', on=unicite_stations)
pdc_stat_unique = pdc_stat.sort_values(by=id_station_pdc + filtre).drop_duplicates(id_station_pdc, keep='last').copy()

# dédoublonnage des pdc suivant son id_pdc avec filtrage par priorite, date_maj, last_modif (priorité déja filtrée sur les stations mais nécessaire)
pdc_direct =  pdc_stat_unique.sort_values(by=[id_pdc] + filtre).drop_duplicates(id_pdc, keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

### Bilan dédoublonnage direct

In [11]:
print("nombre de pdc initial {}, avec dédoublonnage des pdc multi-stations {} et avec dédoublonnage de l'historique {} (supprimés {})\n".format(len(pdc_stat), len(pdc_stat_unique), len(pdc_direct), len(pdc_stat)-len(pdc_direct)))
print(pdc_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_direct = analyse_integrite(pdc_direct, schema)

nombre de pdc initial 172995, avec dédoublonnage des pdc multi-stations 172424 et avec dédoublonnage de l'historique 144823 (supprimés 28172)

datagouv_organization_or_owner
QualiCharge                       61545
GIREVE                            22430
Alizé                             13576
IZIVIA                            12241
Indigo Group                       6907
Eco-Movement                       3912
Driveco                            3460
TotalEnergies Marketing France     1848
Engie Mobilités Electriques        1745
Electric 55 Charging               1291
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                   7
coordonneesXY - id_station_itinerance                964
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 1246
implantation_station - id_station_itinerance          54

### Dédoublonnage indirect station

In [12]:
stations_direct = pdc_direct.drop_duplicates([id_station]).copy()

In [13]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec le champ 'attribut' identique et venant de plusieurs origines
def indirect_station(stations, attribut, affiche=True):
    dupl_att_origine = ~stations.duplicated(keep=False, subset=[attribut, origine])
    
    stations_ext = stations[dupl_att_origine].copy()
    stations_int = stations[~dupl_att_origine].copy()

    filtrage = stations_ext.sort_values(by=[attribut, priorite])
    stat_att = filtrage.drop_duplicates([attribut], keep='last').copy()
    
    duplicates = filtrage.duplicated(subset=[attribut], keep='last')
    #print(duplicates)
    duplicated = stations_ext.loc[duplicates] #.copy()
    
    resultat = pd.concat([stations_int, stat_att])
    if affiche :
        print("nombre de stations dupliquées pour l'attribut '{:<15}' : {}".format(attribut, len(stations_ext) - len(stat_att)))
        print("nombre initial de stations {}, avec dédoublonnage {} {}\n".format(len(stations), attribut, len(resultat)))
    return (resultat, duplicated)

#### Evaluation des stations avec un attribut identique et origine différente
Cette étape donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec un attribut identiques et venant de plusieurs origines

In [14]:
stations_xy, dupl_xy = indirect_station(stations_direct, coord)
stations_nom, dupl_nom = indirect_station(stations_direct, nom_station)
stations_adr, dupl_adr = indirect_station(stations_direct, adresse)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 813
nombre initial de stations 44959, avec dédoublonnage coordonneesXY 44146

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 106
nombre initial de stations 44959, avec dédoublonnage nom_station 44853

nombre de stations dupliquées pour l'attribut 'adresse_station' : 99
nombre initial de stations 44959, avec dédoublonnage adresse_station 44860



#### Dédoublonnage

In [15]:
stations_xy, dupl_xy = indirect_station(stations_direct, coord)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 813
nombre initial de stations 44959, avec dédoublonnage coordonneesXY 44146



In [16]:
stations_nom, dupl_nom = indirect_station(stations_xy, nom_station)

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 46
nombre initial de stations 44146, avec dédoublonnage nom_station 44100



In [17]:
stations_adr, dupl_adr = indirect_station(stations_nom, adresse)

nombre de stations dupliquées pour l'attribut 'adresse_station' : 46
nombre initial de stations 44100, avec dédoublonnage adresse_station 44054



In [18]:
# le gain avec la suppression des doublons d'adresse est faible et on supprime quelques stations différentes, on se limite aux coordonnées et au nom
stations_indirect = stations_nom

In [19]:
stations_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
QualiCharge                                  13597
IZIVIA                                        6611
GIREVE                                        6261
Alizé                                         4165
Eco-Movement                                  3826
Driveco                                        889
Syndicat Départemental d'Energie du Tarn       653
Citeos Ingénierie IdF & Est (Cogelum IdF)      615
ZE-WATT                                        607
SOREGIES                                       396
Electric 55 Charging                           391
ZEborne                                        377
Mobilize Power Solutions                       347
Rossini Energy                                 311
Engie Mobilités Electriques                    309
Name: index, dtype: int64

### Dédoublonnage indirect Qualicharge
origine Qualicharge, même nom, même coordonnées, unité différente

In [32]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec le champ 'attribut' identique et venant de plusieurs origines
def indirect_qualicharge(stations, affiche=True):
    stat_quali = stations[stations[origine]=='QualiCharge'].copy()
    stat_quali['unite'] = stat_quali[id_station].str[:5]
    stat_not_quali = stations[stations[origine]!='QualiCharge'].copy()

    filtrage = stat_quali.sort_values(by=[nom_station, coord])
    duplicates = filtrage.duplicated(subset=[nom_station, coord], keep=False)
    duplicated = stat_quali.loc[duplicates]
    '''dupl_att_origine = ~stat_quali.duplicated(keep=False, subset=[attribut, origine])
    
    stations_ext = stations[dupl_att_origine].copy()
    stations_int = stations[~dupl_att_origine].copy()

    filtrage = stations_ext.sort_values(by=[attribut, priorite])
    stat_att = filtrage.drop_duplicates([attribut], keep='last').copy()
    
    duplicates = filtrage.duplicated(subset=[attribut], keep='last')
    #print(duplicates)
    duplicated = stations_ext.loc[duplicates] #.copy()
    
    resultat = pd.concat([stations_int, stat_att])
    if affiche :
        print("nombre de stations dupliquées pour l'attribut '{:<15}' : {}".format(attribut, len(stations_ext) - len(stat_att)))
        print("nombre initial de stations {}, avec dédoublonnage {} {}\n".format(len(stations), attribut, len(resultat)))'''
    return duplicated.sort_values(by=[nom_station, coord])

In [43]:
stat_indirect_quali = indirect_qualicharge(stations_indirect)
stat_indirect_quali[att_station]

,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY
117450,FRSE1PSE62BBRA,2025-12-15,2026-02-28T00:40:31.000000+0000,STATIONS-E,3BRASSEURS - Bruay la Buissiere,"[2.58027, 50.49518]"
117452,FRSE1PSE62BBRB,2025-12-15,2026-02-28T00:40:31.000000+0000,STATIONS-E,3BRASSEURS - Bruay la Buissiere,"[2.58027, 50.49518]"
117817,FRSE1PSE86CXQA,2025-12-15,2026-02-28T00:40:31.000000+0000,STATIONS-E,ABYSSEA - Civaux,"[0.66189, 46.44501]"
117820,FRSE1PSE86CXQB,2025-12-15,2026-02-28T00:40:31.000000+0000,STATIONS-E,ABYSSEA - Civaux,"[0.66189, 46.44501]"
20082,FREBNPBECLS,2026-02-12,2026-02-28T00:40:31.000000+0000,SPBR1,ALLEVARD - Station Du Collet,"[6.10633, 45.39333]"
...,...,...,...,...,...,...
17011,FRDREP4514745,2026-02-15,2026-02-28T00:40:31.000000+0000,DREAM ENERGY,e-Fast 150kW - Parc commercial a√©roport Poiti...,"[0.31370, 46.58460]"
17002,FRDREP754066,2026-02-15,2026-02-28T00:40:31.000000+0000,DREAM ENERGY,e-Fast 150kW - Village des marques Troyes,"[4.11000, 48.33000]"
97574,FRQWTP754066,2025-07-03,2026-02-28T00:40:31.000000+0000,QoWatt,e-Fast 150kW - Village des marques Troyes,"[4.11000, 48.33000]"
51842,FRIENP040101,2025-07-28,2026-02-28T00:40:31.000000+0000,NW IECharge,Épeigné-sur-Dême-II,"[0.61901, 47.69014]"


### Dédoublonnage indirect pdc

In [20]:
pdc_indirect = stations_indirect[[id_station]].merge(pdc_direct, how='left', on=id_station)

In [21]:
print("nombre de pdc initial {} et avec dédoublonnage {} (supprimés {})\n".format(len(pdc_direct), len(pdc_indirect), len(pdc_direct)-len(pdc_indirect)))
print(pdc_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15], '\n')
res_indirect = analyse_integrite(pdc_indirect, schema)

nombre de pdc initial 144823 et avec dédoublonnage 142208 (supprimés 2615)

datagouv_organization_or_owner
QualiCharge                                  61545
GIREVE                                       20234
Alizé                                        13563
IZIVIA                                       12236
Indigo Group                                  6897
Eco-Movement                                  3911
Driveco                                       3460
TotalEnergies Marketing France                1845
Engie Mobilités Electriques                   1743
Electric 55 Charging                          1291
Qovoltis                                      1198
Citeos Ingénierie IdF & Est (Cogelum IdF)     1118
Lidl                                          1077
e-Totem                                        896
Rossini Energy                                 855
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance  

## Synthèse des dédoublonnages

In [22]:
print('données brutes       : total pdc {:<7}'.format(len(irve_brut.groupby([id_pdc]).count())))
print('solution actuelle    : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(irve), sum(irve['ok']), len(irve) - sum(irve['ok']), res_actuel['index - id_pdc_itinerance']))
print('proposition direct   : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_direct), sum(pdc_direct['ok']), len(pdc_direct) - sum(pdc_direct['ok']), res_direct['index - id_pdc_itinerance']))
print('proposition indirect : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_indirect), sum(pdc_indirect['ok']), len(pdc_indirect) - sum(pdc_indirect['ok']), res_indirect['index - id_pdc_itinerance']))


données brutes       : total pdc 147826 
solution actuelle    : total pdc 148286  avec pdc ok 132857 , pdc avec erreur 15429 dont doublons 816
proposition direct   : total pdc 144823  avec pdc ok 135295 , pdc avec erreur  9528 dont doublons 0
proposition indirect : total pdc 142208  avec pdc ok 138146 , pdc avec erreur  4062 dont doublons 0


## Test des cas de doublons identifiés

- cas 0 : doublon de pdc d'origine différentes sur une même station
- cas 1 : pdc sur deux stations avec identifiants de station différents
- cas 2 : station avec deux origines, des identifiants différents et mêmes coordonnées
- cas 3 : station avec deux origines, des identifiants et des coordonnées différents, des noms identiques
- cas 4 : station avec deux origines, des identifiants et des coordonnées et des noms différents, des adresses identiques
- cas 5 : station Qualicharge décommissionnée partiellement

In [49]:
def test_doublons(irve):
    return {
        'station Tesla de 48 pdc (cas 0)': len(irve[irve[id_station]=='FRTSLP16281'])==48,
        'station Atlante de 16 pdc (cas 1)': (len(irve[irve[id_station]=='FRATLP1136899124034716498']) + len(irve[irve[id_station]=='FRATLPFR01092'])) == 16,
        'pdc Ionity sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRIOYE423408']) == 1,                 
        'stations de 2 pdc (cas 2)': len(irve[irve[coord]=='[-0.00097, 49.32456]'])==2,
        'stations de 12 pdc (cas 2)': len(irve[irve[coord]=='[-0.05677, 48.72293]'])==12,
        'stations de 2 pdc (cas 3)': len(irve[irve[nom_station]=='VALENCE EN POITOU_SALLE DES FETES'])==2,
        'stations de 3 pdc (cas 4)': len(irve[irve[adresse]=='15 Av. Président Georges Pompidou'])==3,
        'station R3 décommissionnée partiellement': len(irve[irve[id_station]=='FRR3MP1063577'])==1
    }
def cumul_tests(resultat):
    return sum(resultat.values())

In [50]:
res_actuel = test_doublons(irve)
propos_direct = test_doublons(pdc_direct)
propos_indirect = test_doublons(pdc_indirect)
sum(res_actuel.values()), sum(propos_direct.values()), sum(propos_indirect.values())

(3, 4, 7)

In [51]:
res_actuel

{'station Tesla de 48 pdc (cas 0)': True,
 'station Atlante de 16 pdc (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 décommissionnée partiellement': False}

In [52]:
propos_indirect

{'station Tesla de 48 pdc (cas 0)': True,
 'station Atlante de 16 pdc (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': True,
 'stations de 12 pdc (cas 2)': True,
 'stations de 2 pdc (cas 3)': True,
 'stations de 3 pdc (cas 4)': False,
 'station R3 décommissionnée partiellement': True}

### Dédoublonnage indirect Qualicharge
origine Qualicharge, même nom, même coordonnées, 

In [44]:
irve[irve[id_pdc]=='FRR3ME5372138'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY
12314,FRR3ME5372138,FRR3MP1063577,2026-01-07,2026-01-08T05:09:01.000000+0000,R3,Villeneuve d'Ascq - Boulangerie Ange,"[50.60601, 3.16438]"


In [47]:
pdc_indirect[pdc_indirect[id_pdc]=='FRR3ME5372138'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY


In [76]:
irve_brut[irve_brut[id_pdc]=='FRWA2E3269222'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY


In [59]:
irve[irve[id_station]=='FREVZPD41C4845EAB44A929A52F584A947D'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY


In [60]:
pdc_indirect[pdc_indirect[id_station]=='FREVZPD41C4845EAB44A929A52F584A947D'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY


In [75]:
irve_brut[irve_brut[id_station]=='FRWA2P1235253'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY


In [68]:
irve_brut.columns

Index(['index', 'nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'consolidated_longitude',
       'consolidated_latitude', 'datagouv_dataset_id', 'datagouv_resource_id',
       'dataset_title', 'datagouv_organization_or_owner',
       'da